# 电气安全答题程序数据分析

本笔记本用于分析电气安全答题程序的运行结果和题库数据。

In [ ]:
# 导入必要的库
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. 读取运行结果

In [ ]:
# 读取运行结果文件
result_path = '../output/run_result.txt'

if os.path.exists(result_path):
    with open(result_path, 'r', encoding='utf-8') as f:
        content = f.read()
    print('文件内容预览：')
    print('='*50)
    print(content)
    print('='*50)
else:
    print(f'错误：未找到文件 {result_path}')

## 2. 解析运行结果

In [ ]:
# 解析得分、答对、答错数量
def parse_result(content):
    score = re.search(r'最终得分:\s*(\-?\d+)', content)
    correct = re.search(r'答对:\s*(\d+)', content)
    wrong = re.search(r'答错:\s*(\d+)', content)
    
    result = {
        'score': int(score.group(1)) if score else None,
        'correct': int(correct.group(1)) if correct else None,
        'wrong': int(wrong.group(1)) if wrong else None
    }
    return result

# 解析结果
result = parse_result(content)
print('解析结果：')
print(f"  最终得分: {result['score']}")
print(f"  答对数量: {result['correct']}")
print(f"  答错数量: {result['wrong']}")

## 3. 可视化分析

In [ ]:
# 创建数据分析图表
plt.figure(figsize=(12, 5))

# 饼图：答对答错比例
plt.subplot(1, 2, 1)
labels = ['答对', '答错']
sizes = [result['correct'], result['wrong']]
colors = ['#28a745', '#dc3545']
explode = (0.05, 0)

plt.pie(sizes, explode=explode, labels=labels, colors=colors,
        autopct='%1.1f%%', shadow=True, startangle=90)
plt.title('答题正确率分布')
plt.axis('equal')

# 柱状图：得分统计
plt.subplot(1, 2, 2)
categories = ['最终得分', '答对题数', '答错题数']
values = [result['score'], result['correct'], result['wrong']]
colors = ['#08669b', '#28a745', '#dc3545']

plt.bar(categories, values, color=colors)
plt.title('答题成绩统计')
plt.ylabel('数量/分数')

# 添加数值标签
for i, v in enumerate(values):
    plt.text(i, v + 0.1, str(v), ha='center')

plt.tight_layout()
plt.show()

## 4. 题库数据分析

In [ ]:
# 定义题库数据
quiz_data = [
    {'category': '触电急救', 'question': '发现有人触电时，首先应该怎么做？', 'answer': 2},
    {'category': '火灾预防', 'question': '以下哪种情况最容易引发电气火灾？', 'answer': 2},
    {'category': '用电安全', 'question': '湿手为什么不能触摸电器开关？', 'answer': 3},
    {'category': '火灾处理', 'question': '发现电器着火，首先应该？', 'answer': 3},
    {'category': '用电习惯', 'question': '以下哪种行为是正确的用电习惯？', 'answer': 3},
    {'category': '雷电防护', 'question': '遇到雷雨天，以下哪种行为是错误的？', 'answer': 3},
    {'category': '电路保护', 'question': '家庭电路保险丝的作用是？', 'answer': 2},
    {'category': '灭火知识', 'question': '以下哪种物品不能作为电气火灾灭火器使用？', 'answer': 3},
    {'category': '应急处理', 'question': '如果看到电线断落在地上，应该怎么做？', 'answer': 3},
    {'category': '接地保护', 'question': '以下关于接地线的说法，正确的是？', 'answer': 2},
    {'category': '电器使用', 'question': '使用移动电器时，应该注意什么？', 'answer': 2},
    {'category': '安全规范', 'question': '什么是电气安全的"三做到"？', 'answer': 2},
    {'category': '火灾逃生', 'question': '发生电气火灾时，正确的逃生方法是？', 'answer': 2},
    {'category': '临时用电', 'question': '临时用电应该注意什么？', 'answer': 2},
    {'category': '安全电压', 'question': '安全电压是指多少伏以下的电压？', 'answer': 2}
]

# 转换为DataFrame
df = pd.DataFrame(quiz_data)
print('题库数据预览：')
display(df.head())

In [ ]:
# 统计各分类题目数量
category_counts = df['category'].value_counts()

print('各分类题目数量统计：')
print(category_counts)

# 可视化
plt.figure(figsize=(10, 6))
sns.barplot(x=category_counts.values, y=category_counts.index, palette='viridis')
plt.title('题库分类分布')
plt.xlabel('题目数量')
plt.ylabel('分类')
plt.show()

## 5. 发现的问题分析

In [ ]:
# 分析发现的问题
issues = [
    {'问题编号': 'ISSUE-001', '问题描述': '题库分类不均衡，部分分类题目过多', '严重程度': '中等', '建议': '增加其他分类的题目，使分布更均衡'},
    {'问题编号': 'ISSUE-002', '问题描述': '分数可能出现负数', '严重程度': '低', '建议': '设置最低分数为0，避免负分'},
    {'问题编号': 'ISSUE-003', '问题描述': '选项编号固定为1-4，缺少随机打乱', '严重程度': '低', '建议': '每次答题时随机打乱选项顺序'},
    {'问题编号': 'ISSUE-004', '问题描述': '缺少答题历史记录功能', '严重程度': '低', '建议': '添加答题记录保存功能'},
    {'问题编号': 'ISSUE-005', '问题描述': '答对时缺少正向反馈', '严重程度': '低', '建议': '添加答对时的鼓励性提示'}
]

issues_df = pd.DataFrame(issues)
print('发现的问题列表：')
display(issues_df)

## 6. 结论与建议

In [ ]:
# 生成分析报告
print('='*60)
print('           电气安全答题程序分析报告')
print('='*60)

# 运行结果摘要
print('\n【运行结果摘要】')
print(f"  最终得分: {result['score']} 分")
print(f"  答对题数: {result['correct']} 道")
print(f"  答错题数: {result['wrong']} 道")
print(f"  正确率: {result['correct']/5*100:.1f}%")

# 题库分析
print('\n【题库分析】')
print(f"  题库总题数: {len(df)} 道")
print(f"  分类数量: {len(category_counts)} 个")

# 改进建议
print('\n【改进建议】')
print('  1. 增加题库分类多样性')
print('  2. 优化分数计算逻辑')
print('  3. 添加选项随机打乱功能')
print('  4. 增加答题记录功能')

print('\n' + '='*60)